# Spatial Proteomics Analysis (SpPrAn)

Welcome to our pipeline's example!    
This notebook is intended to show you how to execute the pipeline or specific functions in a Jupyter Notebook, as well as to visualize resulting plots.

It is important to mention that for executing this notebook, you need to create a copy into repository's root directory.

## Configurations for pipeline

The first thing we suggest to do is adjusting the pipeline configurations to your needs.   
Specifically speaking, you need to adjust the following:

1. **Workspace**: Here, you need to establish the ***input directory*** (where your COMET™ data is located), the ***filetype*** of your COMET™ data (usually is *tsv*, but also pipeline accepts *csv*), and the ***output directory*** (where the results from the pipeline will be saved).

2. Adding a list of **protein markers** that will be used for cell typing

3. Adding a dictionary for estabishing the **cell types** and which protein markers should be listen for defining them.

4. Adding a dictionary of **custom colors** for identifying the cell types

5. Other configurations related to the pipeline

In order to do that, we suggest to duplicate the YAML file located in `config/config_example.yaml` and then edit it for your specific conditions.

In [ ]:
!cp config/config_example.yaml config/config.yaml

Then, you can open `config.yaml` in the code editor of your preference (we suggest **VS Code**) for editing.  
This is how `config.yaml` file looks like:

```
# Add the pathnames to the directories inside quotes, as well as the files' format type (tsv or csv).
workspace:
  input_dir: "/data/example/input"
  output_dir: "/data/example"
  filetype: tsv

# Protein markers should be named exactly as in the "Positivity" columns in the csv files.
# To add a new protein marker, simply add a new line following the format of the previous ones,
#  without forgetting that the protein marker's name must be inside quotes.
protein_markers:
  - "Positivity - Pan-CK* (MV - CYTO)"      # first element
  - "Positivity - CD45* (MV - CYTO)"        # second element
  - "Positivity - Vimentin* (MV - CYTO)"    # third element
  - "Positivity - CD31* (MV - CYTO)"        # ...

# For defining cell types: 
# - Name all the cell types using quotes.
# - For each cell type, generate a list containing only 0, 1, and ~ values.
# - Lists corresponds to the previous protein_markers list and resemble their presense or absent in
#    the defined type of cells.
# - 1 means is protein marker is present, 0 means is abscent, ~ means is not being consider as part of
#    the definition for this cell type.
# - Lists MUST have the same amount of values as proteins in the protein_markers list.
# - To add a new cell type, simply add a new line following the format of the previous ones.
cell_types:
  "C cells": [0, 0, 1, 0]  # [first element, second element, third element, ...]
  "T1 cells": [1, 0, 0, 0]
  "T2 cells": [1, 0, 1, 0]
  "I cells": [0, 1, ~, 0]

# If needed, please establish the HEX color code for each cell type. If not needed, comment or delete this option.
# Cell type's name MUST be the same as in cell_types.
# It MUST contain an "Other cells" name with its corresponding HEX color value.
# Cell type's names in custom_colors MUST be ordered in descending order.
custom_colors:
  "C cells": "#00aded"
  "I cells": "#ff0000"
  "Other cells": "#a3a3a3"
  "T1 cells": "#fbfb00"
  "T2 cells": "#00ff00"

# Setting this value to True will overwrite previously output files, such as csv files and plots.
overwrite_existing_files: False

# Setting this value to False will not locally save anndata files and it will require to reprocess
#  information each time pipeline is run.
locally_save_anndata_files: True

# Spatial plot adjustments
spatial_plot:
  dpi: 300
  scatter_point_size: 100
```

You can also edit `config.yaml` content using Python code.

In [1]:
import yaml

# Read configuration file
with open('config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)
config

{'workspace': {'input_dir': './data/example/input/',
  'output_dir': './data/example/',
  'filetype': 'tsv'},
 'protein_markers': ['Positivity - Pan-CK* (MV - CYTO)',
  'Positivity - CD45* (MV - CYTO)',
  'Positivity - Vimentin* (MV - CYTO)',
  'Positivity - CD31* (MV - CYTO)'],
 'cell_types': {'C cells': [0, 0, 1, 0],
  'T1 cells': [1, 0, 0, 0],
  'T2 cells': [1, 0, 1, 0],
  'I cells': [0, 1, None, 0]},
 'custom_colors': {'C cells': '#00aded',
  'I cells': '#ff0000',
  'Other cells': '#a3a3a3',
  'T1 cells': '#fbfb00',
  'T2 cells': '#00ff00'},
 'overwrite_existing_files': False,
 'locally_save_anndata_files': True,
 'spatial_plot': {'dpi': 300, 'scatter_point_size': 100}}

In [ ]:
# Modify configuration settings to your needs
config['workspace']['input_dir'] = 'data/example/input/'
config['workspace']['output_dir'] = 'data/example/'
config['workspace']['filetype'] = 'tsv'
config['protein_markers'] = [
    'Positivity - panCK* (MV Cell)',
    'Positivity - CD45hu* (MV Cell)',
    'Positivity - Vim* (MV Cell)',
    'Positivity - FAPa* (MV Cell))'
]
config['cell_types'] = {
    'C cells': [0, 0, 1, 0],
    'T1 cells': [1, 0, 0, 0],
    'T2 cells': [1, 0, 1, 0],
    'I cells': [0, 1, None, 0]
}
config['custom_colors'] = {
    'C cells': '#00aded',
    'I cells': '#ff0000',
    'Other cells': '#a3a3a3',
    'T1 cells': '#fbfb00',
    'T2 cells': '#00ff00'
}
config['overwrite_existing_files'] = False
config['locally_save_anndata_files'] = True
config['spatial_plot'] = {
    'dpi': 300,
    'scatter_point_size': 30
}

In [11]:
# Write modified configuration back to file
with open('config/config.yaml', 'w') as f:
    yaml.dump(config, f, sort_keys=False)

## Running the pipeline

Once the YAML file is set with your data, you can execute the pipeline as follows:

Note for me:  
* I need to clean the objects.tsv file, as well as change the way file is read, as well as sample name appearing in file.
* Handle DAPI not found.

In [21]:
!python spatial_proteomics/core.py --config config/config.yaml

/Users/sergio/Documents/Fox Chase/Lee_Lab/VENV/lee_venv/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/sergio/Documents/Fox Chase/Lee_Lab/VENV/lee_venv/lib/python3.12/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
Generating anndata files for analysis...
/Users/sergio/Documents/Fox Chase/Lee_Lab/VENV/lee_venv/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
Anndata generated!
Anndata saved!
/Users/sergio/Documents/Fox Chase/Lee_La

In [14]:
import pandas as pd
filename = []
for file in filename:
    temp = pd.read_csv(f"{file}")
temp

NameError: name 'temp' is not defined

# Other

Here, we are loading the all the pipeline functions, as well as other Python libraries will need for manipulating outputs.

In [ ]:
# from pathlib import Path                                # used
# import os                                               # used
# import re

# import numpy as np                                      # used
# np.float_ = np.float64                                  # used
# import pandas as pd                                     # used
# from anndata import AnnData                             # used

# import scanpy as sc                                     # used
# import squidpy as sq                                    # used

# import matplotlib.pyplot as plt                         # used
# from matplotlib import rcParams                         # used
# from matplotlib.colors import ListedColormap            # used
# import seaborn as sns                                   # used

# from scipy.stats import ttest_ind                       # used
# from statsmodels.stats.multitest import multipletests   # used
# from scipy.stats import mannwhitneyu                    # used



## Function for loading adata files

In [2]:
def load_anndata_files():
    sample_names = pd.read_csv(f"{results_dir}Samples Id.csv")['Samples Id'].tolist()
    adata_dicts = {}
    for sample in sample_names:
        adata_dicts[sample] = sc.read_h5ad(f"{adata_dir}Sample_{sample}_after_umap.h5ad")
    return adata_dicts

## General settings

In [ ]:
# sc.settings.autoshow = False
# sc.settings.verbosity = 3  # for hints
# sc.logging.print_header()
# sc.settings.set_figure_params(dpi_save=300, dpi=80, facecolor="white")
# rcParams["figure.figsize"] = (10,10)

2025-12-19 09:06:11.453983: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-19 09:06:11.493267: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affec

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/usr/local/lib/python3.10/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.10/dist-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/local/lib/python3.10/dist-packages/ipykernel/kernelapp.p

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/usr/local/lib/python3.10/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.10/dist-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/local/lib/python3.10/dist-packages/ipykernel/kernelapp.p

AttributeError: _ARRAY_API not found

ImportError: numpy.core._multiarray_umath failed to import

scanpy==1.10.2 anndata==0.10.8 umap==0.5.6 numpy==2.0.0 scipy==1.14.0 pandas==2.2.2 scikit-learn==1.5.0 statsmodels==0.14.2 igraph==0.11.5 pynndescent==0.5.13


## Setting workspace directory

In this cell, please establish the directory where your files are and where your files will be stored.

In [4]:
# USER CAN MODIFY
workspace_dir = f"../hPDAC_TMA_24-15/test/"
results_dir   = f"{workspace_dir}results/"
files_dir     = f"../hPDAC_TMA_24-15/files/new/"
filetype      = "tsv"
plots_dir     = f"{workspace_dir}plots/"
adata_dir     = f"{workspace_dir}adata/"

This code is to obtain the filenames of the samples we are going to analyze.

In [5]:
path = Path(files_dir)
entries = list(path.iterdir())
filenames = [entry for entry in entries if entry.is_file() and entry.match(f"*objects.{filetype}")]

## Main settings for user

In this cell, please establish cell types to study, as well as the protein markers and their presence or absence that defines the cell types.

Protein markers should be named exactly as in the csv positivity column.

In [6]:
# USER CAN MODIFY
protein_markers = [
    "Positivity - Pan-CK* (MV - CYTO)", # element 0
    "Positivity - CD45* (MV - CYTO)",   # element 1
    "Positivity - Vimentin* (MV - CYTO)",
    "Positivity - CD31* (MV - CYTO)",
]

# USER CAN MODIFY
# protein markers: 1 is present, 0 is abscent.
cell_types = {
    "CAF cells":     {protein_markers[0]: 0, protein_markers[1]: 0, protein_markers[2]: 1, protein_markers[3]: 0},
    "Immune cells":  {protein_markers[0]: 0, protein_markers[1]: 1,                        protein_markers[3]: 0},
    "Tumor cells 1": {protein_markers[0]: 1, protein_markers[1]: 0, protein_markers[2]: 0, protein_markers[3]: 0},
    "Tumor cells 2": {protein_markers[0]: 1, protein_markers[1]: 0, protein_markers[2]: 1, protein_markers[3]: 0},
}

In this cell, please establish the HEX color code for each cell type if needed.

In [7]:
# USER CAN MODIFY
custom_colors = {
    "CAF cells"    : "#00aded",
    "Immune cells" : "#ff0000",
    "Other cells"  : "#a3a3a3",
    "Tumor cells 1": "#fbfb00",
    "Tumor cells 2": "#00ff00",
}

In this cell, please adjust values for UMAP if needed

In [8]:
# USER CAN MODIFY
random_state = 160888
n_neighbors  = 40
resolution   = 0.5

In this cell, please establish if you want to plot the corresponding pie plots (True) or not (False)

In [9]:
# USER CAN MODIFY
want_to_plot_pie_plots=True

In this cell, please establish if you want to replot again all plots (and generate csv files). WARNING: "True" value will delete all previous plots.

In [10]:
# USER CAN MODIFY
override_existing_plots = False

In this cell, please adjust protein markers for obtaining the subpopulation of the interested cell type cells.

In [11]:
# # USER CAN MODIFY - CAF
# ct_name = "CAF cells"  # Please, verify that this name is the same as introduced in cell_types dictionary.
# short_ct_name = "CAF"
# subCT_markers = [   # Here, put the protein markers' name that is contained in the csv positivity column.
#     "P-FAK",
#     "Netrin G1",
#     "FAPa",
#     "Pall-iso3",
#     "P-SMAD2",
#     "P-TIE2* (MV - CYTO)",
#     "aSMA",
#     "PD-L1",
#     "CTLA4",
#     "NGL-1",
#     "Podoplanin",
#     "P-TIE2* (MV - NUC)",
#     "CLIC3",
#     "Ki67",
#     "53BP1"
# ]
# initials_sCT = [  # Here, put the protein markers' initials for making the name shorter and for the later identification.
#     "Pf",
#     "Ne",
#     "F",
#     "Pa",
#     "Ps",
#     "Ptc",
#     "A",
#     "Pd",
#     "Ct",
#     "Ng",
#     "Po",
#     "Ptn",
#     "Cl",
#     "K",
#     "5"
# ]

In [12]:
# USER CAN MODIFY - Immune
ct_name = "Immune cells"  # Please, verify that this name is the same as introduced in cell_types dictionary.
short_ct_name = "Immune"
subCT_markers = [   # Here, put the protein markers' name that is contained in the csv positivity column.
    'FOXP3','CD8', 'CD11c','CD20','CD3','CD4','CD56','CD68','Granzyme-B', 'PD-1', 'PD-L1', 'CTLA4', 'Ki67','53BP1'
]
initials_sCT = [  # Here, put the protein markers' initials for making the name shorter and for the later identification.
    'FOXP3','CD8', 'CD11c','CD20','CD3','CD4','CD56','CD68','Gran','PD1', 'PDL1', 'CTLA4', 'Ki67','53BP1'
]

In [13]:
# # USER CAN MODIFY
# ct_name = "Tumor cells 1"  # Please, verify that this name is the same as introduced in cell_types dictionary.
# short_ct_name = "Tumor1"
# subCT_markers = [   # Here, put the protein markers' name that is contained in the csv positivity column.
#     'P-SMAD2','P-FAK', 'Pall-iso3','NGL-1','Netrin G1','Ki67','53BP1','PD-L1', 'CLIC3', 'CD56',
# ]
# initials_sCT = [  # Here, put the protein markers' initials for making the name shorter and for the later identification.
#     'Ps','Pf', 'Pa','Ng','Ne','Ki','53','Pd', 'Cl', 'CD56',
# ]

In [14]:
# # USER CAN MODIFY
# ct_name = "Tumor cells 2"  # Please, verify that this name is the same as introduced in cell_types dictionary.
# short_ct_name = "Tumor2"
# subCT_markers = [   # Here, put the protein markers' name that is contained in the csv positivity column.
#     'P-SMAD2','P-FAK', 'Pall-iso3','NGL-1','Netrin G1','Ki67','53BP1','PD-L1', 'CLIC3', 'CD56',
# ]
# initials_sCT = [  # Here, put the protein markers' initials for making the name shorter and for the later identification.
#     'Ps','Pf', 'Pa','Ng','Ne','Ki','53','Pd', 'Cl', 'CD56',
# ]

In this cell, please establish which samples corresponds to "Control" and which ones to "Treatment".

In [15]:
# USER CAN MODIFY
dict_ctrl = {
    "D2":"TN",
    "D3":"TN",
    "D4":"TN",
    "D5":"TN",
    "E2":"TN",
    "E3":"TN",
    "E4":"TN",
    "F2":"TN",
    "F3":"TN",
    "F4":"TN",
    "F5":"TN",
    "G2":"TN",
    "G4":"TN"
}
dict_trtmnt = {
    "B2":"TNT",
    "B3":"TNT",
    "B4":"TNT",
    "B5":"TNT",
    "B6":"TNT",
    "C2":"TNT",
    "C3":"TNT",
    "C4":"TNT",
    "C5":"TNT",
    "D6":"TNT",
    "D7":"TNT",
    "E6":"TNT",
    "E7":"TNT"
}

# If data has not ever being processed, start here:

## QC

First, we are going to filter out cells without nucleus and keep all the the features we want to hold (protein markers and spatial information) for all the samples in filenames.

This function will return two dictionaries:
1. A dictionary with all cleaned data in DataFrame (pandas) format.
2. A dictionary with all cleaned data in AnnData format.

Please, make sure that 1 means that protein marker is present and 0 means that protein marker is abscent in "Positivity" columns. 

In [15]:
def cleaned_data(file_names, files_dir, filetype):
    adata_dicts = {}
    data_dicts  = {}
    for filename in file_names:
        separator = '\t' if filetype=='tsv' else ','
        data = pd.read_csv(f"{filename}",sep=separator)
        data = data[data['Positivity - DAPI (MV - NUC)']==1] # filter out cells without nucleus
        
        temp = pd.concat([data.iloc[:,3], data.iloc[:, 12:]], axis=1)                               # retain only "Name" and data columns
        temp['Name'] = [f"{s[-5]}{s[-2]}_{i:05}" for s,i in zip(temp['Name'],temp['Name'].index)]   # retain letter and number for identifying samples
        temp.rename(columns={"Name":"cellID"}, inplace=True)
        
        pos_cols = [col for col in temp.columns if "Positivity" in col and "(MV" in col]            # identify columns that says if protein marker is present (1) or abscent (0)
        data = temp[~(temp[pos_cols].eq(-1).any(axis=1))]                                           # filter out those cells that does contain a NaN value (-1) in the previous columns
        columns_nuc = data.columns[                                                                 # keep columns with protein intensity in the nucleus
            (data.columns.isin(['cellID']))|
            ((data.columns.str.contains("MV - NUC - "))&(~data.columns.str.contains("Type")))
        ]
        columns_pos = data.columns[data.columns.isin(['cellID', 'X-coordinate', 'Y-coordinate'])]   # keep spatial columns
        
        adata = AnnData(                                                                            # generate AnnData file to include spatial data
            data[columns_nuc].iloc[:,1:],
            obsm={
                "spatial": data[columns_pos].iloc[:,1:].to_numpy(),
                "ID_cell":data[['cellID']].to_numpy(),
                "Positivity":data[pos_cols].to_numpy()
            }
        )
        adata_dicts[f"{data.iloc[0,0][:2]}"] = adata
        data_dicts[f"{data.iloc[0,0][:2]}"] = data
    df = pd.DataFrame({"Samples Id": list(data_dicts.keys())})
    df.to_csv(f"{results_dir}Samples Id.csv", index=False)
    df = pd.DataFrame({"Positivity column names": list(data[pos_cols].columns)})
    df.to_csv(f"{results_dir}Positivity column names.csv", index=False)
    return data_dicts, adata_dicts

This code is for applying the previous function.

In [16]:
data_dicts, adata_dicts = cleaned_data(file_names=filenames, files_dir=files_dir, filetype=filetype)

/opt/modules/squidpy/squidpy_env/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/opt/modules/squidpy/squidpy_env/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/opt/modules/squidpy/squidpy_env/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/opt/modules/squidpy/squidpy_env/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/opt/modules/squidpy/squidpy_env/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: Implic

## Adding cell type labels

This function will assing cell types according to their definitions

In [17]:
def assign_cell_type(row):
    for cell_type, rule in cell_types.items():
        if all(row[m] == v for m, v in rule.items()):
            return str(cell_type)
    return "Other cells"

This function labels cell types for creating single (a cell types vs others cells) or multiple (all cell types vs other cells) clusters using the previous assignment.

In [18]:
def labeling_cell_types(data_dicts, adata_dicts, cell_type_dict):
    for (k_data, data), (k, adata) in zip(data_dicts.items(), adata_dicts.items()):
        adata.obs['clusters'] = data.apply(assign_cell_type, axis=1).tolist()
        for cell_type, _ in cell_type_dict.items():
            adata.obs[f"only {cell_type}"] = [t if t==cell_type else "Other cells" for t in adata.obs['clusters']]
        adata_dicts[k] = adata
    return adata_dicts

This code is for applying the previous function.

In [19]:
adata_dicts = labeling_cell_types(data_dicts=data_dicts, adata_dicts=adata_dicts, cell_type_dict=cell_types)

## UMAP calculation

This function is for calculating UMAP

In [20]:
def calculate_UMAP(adata_dicts,random_state=160888,n_neighbors=40,resolution=0.5):
    for k, adata in adata_dicts.items():
        sc.pp.normalize_total(adata)
        sc.pp.pca(adata,random_state=random_state)
        sc.pp.neighbors(adata, n_neighbors=n_neighbors,random_state=random_state)
        sc.tl.umap(adata, random_state=random_state)
        sc.tl.leiden(adata, resolution=resolution)
        adata_dicts[k] = adata
    return adata_dicts

This function is for saving adata locally

(since pp.neighbors and tl.umap are time-consuming processes and it is whortless to calculate them every time just for plotting)

In [21]:
def save_anndata_files(adata_dicts):
    for k, adata in adata_dicts.items():
        adata.write(f"{adata_dir}Sample_{k}_after_umap.h5ad")

These codes are for applying the previous function.

In [22]:
adata_dicts = calculate_UMAP(adata_dicts,random_state=random_state,n_neighbors=n_neighbors,resolution=resolution)

normalizing counts per cell
    finished (0:00:00)
computing PCA
    with n_comps=31
    finished (0:00:00)
computing neighbors
    using data matrix X directly


/opt/modules/squidpy/squidpy_env/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:02)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm) (0:00:01)
running Leiden clustering


/tmp/ipykernel_1769811/2045822333.py:7: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=resolution)


    finished: found 7 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:00)
normalizing counts per cell
    finished (0:00:00)
computing PCA
    with n_comps=31
    finished (0:00:00)
computing neighbors
    using data matrix X directly
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:00)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm) (0:00:03)
running Leiden clustering
    finished: found 8 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:00)
normalizing counts per cell
    finished (0:00:00)
computing PCA
    with n_comps=31
    finished (0:00:00)
computing neighbors
    using data matrix X directly
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:00)

In [23]:
save_anndata_files(adata_dicts)

# If data has been processed and just want to plot it, start here:

## UMAP plot

This function is for calling back the local adata

In [15]:
def load_anndata_files():
    sample_names = pd.read_csv(f"{results_dir}Samples Id.csv")['Samples Id'].tolist()
    adata_dicts = {}
    for sample in sample_names:
        adata_dicts[sample] = sc.read_h5ad(f"{adata_dir}Sample_{sample}_after_umap.h5ad")
    return adata_dicts

This function is for plotting UMAP by SELECTED coloring

In [16]:
def plot_UMAP(adata_dicts,color='leiden',custom_colors=None, dpi=300, bbox_inches='tight'):
    if color=="leiden":
        for k, adata in adata_dicts.items():
            title_name = f"Sample {k} ({adata.n_obs} cells)"
            save_namefile = f"{plots_dir}UMAP leiden - {title_name}.png"
            if os.path.exists(save_namefile) and not override_existing_plots: continue
            sc.pl.umap(adata, color=color, palette=custom_colors, show=False)
            plt.savefig(save_namefile, dpi=dpi, bbox_inches=bbox_inches)
            plt.close()
    elif color=="clusters":
        for k, adata in adata_dicts.items():
            title_name = f"Sample {k} ({adata.n_obs} cells)"
            save_namefile = f"{plots_dir}UMAP cell type - {title_name}.png"
            if os.path.exists(save_namefile) and not override_existing_plots:
                for cell_type,_ in custom_colors.items():
                    if cell_type=='Other cells': continue
                    title_name = f"Sample {k} - {cell_type} ({adata.n_obs} cells)"
                    save_namefile = f"{plots_dir}UMAP cell type - {title_name}.png"
                    if os.path.exists(save_namefile) and not override_existing_plots: continue
                    order_list = [cell_type,'Other cells'] if cell_type<'Other cells' else ['Other cells',cell_type]
                    very_custom_color = {key: custom_colors[key] for key in order_list if key in custom_colors}
                    sc.pl.umap(adata, color=f"only {cell_type}", palette=very_custom_color, show=False)
                    plt.savefig(save_namefile, dpi=dpi, bbox_inches=bbox_inches)
                    plt.close()
                continue
            sc.pl.umap(adata, color=color, palette=custom_colors, show=False)
            plt.savefig(save_namefile, dpi=dpi, bbox_inches=bbox_inches)
            plt.close()
            for cell_type,_ in custom_colors.items():
                if cell_type=='Other cells': continue
                title_name = f"Sample {k} - {cell_type} ({adata.n_obs} cells)"
                save_namefile = f"{plots_dir}UMAP cell type - {title_name}.png"
                if os.path.exists(save_namefile) and not override_existing_plots: continue
                order_list = [cell_type,'Other cells'] if cell_type<'Other cells' else ['Other cells',cell_type]
                very_custom_color = {key: custom_colors[key] for key in order_list if key in custom_colors}
                sc.pl.umap(adata, color=f"only {cell_type}", palette=very_custom_color, show=False)
                plt.savefig(save_namefile, dpi=dpi, bbox_inches=bbox_inches)
                plt.close()
    else:
        print("Incorrect cluster name. Please verify.")

These codes are for applying the previous function.

In [17]:
try:
    if len(adata_dicts)==0: adata_dicts = load_anndata_files()
except NameError:
    adata_dicts = load_anndata_files()

In [18]:
plot_UMAP(adata_dicts=adata_dicts, color='leiden')

In [19]:
plot_UMAP(adata_dicts=adata_dicts, color='clusters', custom_colors=custom_colors)

## Spatial plot

This function is for plotting spatial data

In [20]:
def plot_spatial(adata_dicts,custom_colors,dpi=300,size=100):
    for k, adata in adata_dicts.items():
        title_name = f"Sample {k} ({adata.n_obs} cells)"
        save_namefile = f"{plots_dir}Spatial - {title_name}.png"
        if os.path.exists(save_namefile) and not override_existing_plots:
            for cell_type, selected_color in custom_colors.items():
                if cell_type=='Other cells': continue
                if (adata.obs['clusters'] == cell_type).sum()==0: continue
                
                title_name = f"Sample {k} - {cell_type} ({adata.n_obs} cells)"
                save_namefile = f"{plots_dir}Spatial - {title_name}.png"
                if os.path.exists(save_namefile) and not override_existing_plots: continue
                
                color_spatial = f"only {cell_type}"
                cmap_gene = selected_color
                own_palette_list = [selected_color,custom_colors['Other cells']] if cell_type<'Other cells' else [custom_colors['Other cells'],selected_color]
                own_palette = ListedColormap(own_palette_list)
                cmap_gene = ListedColormap(cmap_gene)
                
                fig, ax = plt.subplots()
                sq.pl.spatial_scatter(
                    adata, 
                    shape=None, 
                    color=color_spatial,
                    title=title_name,
                    dpi=dpi,
                    cmap=cmap_gene,
                    palette=own_palette,
                    size=size,
                    ax=ax)
                ax.set_facecolor("black")
                ax.set_aspect('equal')
                fig.tight_layout()
                plt.savefig(save_namefile)
                plt.close()
            continue
        
        color_spatial = "clusters"
        cmap_gene = None
        which_colors = []
        for cell_type,selected_color in custom_colors.items():
            if (adata.obs['clusters'] == cell_type).sum()!=0: which_colors.append(selected_color)
        own_palette_list = (which_colors)
        own_palette = ListedColormap(own_palette_list)
        
        fig, ax = plt.subplots()
        sq.pl.spatial_scatter(
            adata, 
            shape=None, 
            color=color_spatial,
            title=title_name,
            dpi=dpi,
            cmap=cmap_gene,
            palette=own_palette,
            size=size,
            ax=ax)
        ax.set_facecolor("black")
        ax.set_aspect('equal')
        fig.tight_layout()
        plt.savefig(save_namefile)
        plt.close()
        
        for cell_type, selected_color in custom_colors.items():
            if cell_type=='Other cells': continue
            if (adata.obs['clusters'] == cell_type).sum()==0: continue
            
            title_name = f"Sample {k} - {cell_type} ({adata.n_obs} cells)"
            save_namefile = f"{plots_dir}Spatial - {title_name}.png"
            if os.path.exists(save_namefile) and not override_existing_plots: continue
            
            color_spatial = f"only {cell_type}"
            cmap_gene = selected_color
            own_palette_list = [selected_color,custom_colors['Other cells']] if cell_type<'Other cells' else [custom_colors['Other cells'],selected_color]
            own_palette = ListedColormap(own_palette_list)
            cmap_gene = ListedColormap(cmap_gene)
            
            fig, ax = plt.subplots()
            sq.pl.spatial_scatter(
                adata, 
                shape=None, 
                color=color_spatial,
                title=title_name,
                dpi=dpi,
                cmap=cmap_gene,
                palette=own_palette,
                size=size,
                ax=ax)
            ax.set_facecolor("black")
            ax.set_aspect('equal')
            fig.tight_layout()
            plt.savefig(save_namefile)
            plt.close()

This code is for applying the previous function

In [21]:
plot_spatial(adata_dicts=adata_dicts,custom_colors=custom_colors)

## Cell type proportions files and plots

This function is for plotting pie plots for cell type proportions

In [22]:
def pct_dists(len_sizes):            # function for obtaining an equally distributed spaces for percentages in slices of pie plot
    splitting_number = (1.05-(1/len_sizes))/(len_sizes-1)
    distances_list = np.round(np.arange(1/len_sizes, 1.05+.01,splitting_number),2).tolist()
    just_because = len(distances_list)
    return [distances_list[just_because-1-idx] for idx in range(just_because)]

def move_pct(t, d):                  # function for moving the percentages according to the previous distributions
    xi, yi = t.get_position()
    ri = np.sqrt(xi**2 + yi**2)
    phi = np.arctan2(yi, xi)
    x = d * ri * np.cos(phi)
    y = d * ri * np.sin(phi)
    t.set_position((x, y))

def plot_pie_plots(sum_dict,custom_colors,title_name):
    save_namefile = f"{plots_dir}{title_name} - Pie plot.png"
    if os.path.exists(save_namefile) and not override_existing_plots: return
    
    fig, ax = plt.subplots(figsize=(8, 8), constrained_layout=True)
    
    k1,_ = next(iter(custom_colors.items()))                            # select first cell type
    if sum_dict['Total cells'] > 0 and sum_dict[k1] > 0:                # first cell type will be the "mouth" of this 'pac-man' plot if it is 'open'
        blue_frac = sum_dict[k1] / sum_dict['Total cells']
        half_blue_angle_deg = (blue_frac * 360) / 2.0
        startangle = half_blue_angle_deg
    else:
        startangle = 90
        
    colors = [color for _,color in custom_colors.items()]
    sizes  = [sum_dict[cell_type] for cell_type,_ in custom_colors.items()]
    pctdists1 = pct_dists(len(sizes))                                   # here, we are equally distributing labels across slices of pie plot

    wedges, texts, autotexts = ax.pie(
        sizes,
        colors=colors,
        startangle=startangle,
        counterclock=False,
        autopct=(lambda p: f'{p:.2f}%') if sum_dict['Total cells'] else None,
        pctdistance=0.75,
        wedgeprops=dict(linewidth=1, edgecolor='white')
    )

    # ax.set_title(f"All cells in {title_name}")
    for t in texts: t.set_visible(False)
    for t, d in zip(autotexts, pctdists1): move_pct(t, d)              # here, we are putting labels using pct dists
    ax.axis('equal')

    ax.legend(
        handles=wedges,
        labels = [f"{cell_type} ({sum_dict[cell_type]})" for cell_type,_ in custom_colors.items()],
        loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False
    )
    
    plt.savefig(save_namefile)
    plt.close()

This function is for calculating the cell proportions and save it in a csv file.

In [23]:
def calculate_cell_proportions(adata_dicts,custom_colors,pie_plot=False):
    for k, adata in adata_dicts.items():
        title_name = f"Sample {k} ({adata.n_obs} cells)"
        save_namefile = f"{results_dir}Cell type proportions - {title_name}.csv"
        if os.path.exists(save_namefile) and not override_existing_plots: continue
            
        sum_dict = {}
        sum_dict["Total cells"] = adata.n_obs
        if sum_dict["Total cells"]==0: continue
        prop_dict = {}
        for cell_type,_ in custom_colors.items():
            sum_dict[cell_type]  = (adata.obs['clusters'] == cell_type).sum()
            prop_dict[cell_type] = str(np.round(sum_dict[cell_type]/sum_dict["Total cells"]*100,2))+"%"
        
        df = pd.DataFrame(data = {cell_type:[sum_dict[cell_type],prop_dict[cell_type]] for cell_type,_ in custom_colors.items()}, index = ["Total cells in cell type", "Percentage"])
        df.to_csv(f"{results_dir}Cell type proportions - {title_name}.csv", index=False)
        
        if pie_plot: plot_pie_plots(sum_dict,custom_colors,title_name)

This code is for applying the previous function

In [24]:
calculate_cell_proportions(adata_dicts=adata_dicts,custom_colors=custom_colors,pie_plot=want_to_plot_pie_plots)

# Cell type phenotype analysis

In this section, is introduced the code for obtaining the subpopulation of a cell type by protein markers.

## Subpopulation of a cell type cells

This code is for loading processed data

In [16]:
try:
    if len(adata_dicts)==0: adata_dicts = load_anndata_files()
except NameError:
    adata_dicts = load_anndata_files()

This function is for naming sub cell types (denoted by subCT) by amount of protein markers and initials.

In [17]:
def naming_subCTs(subCT_markers,initials_sCT,short_ct_name):
    how_much_subCT = 2**len(subCT_markers)
    df_bin = pd.DataFrame(data=[n for n in range(how_much_subCT)],columns=["number"])
    df_bin['binary'] = [bin(n)[2:].zfill(len(subCT_markers)) for n in range(how_much_subCT)]              # Binary id
    df_bin['list_pos'] = [list(bin(n)[2:].zfill(len(subCT_markers))) for n in range(how_much_subCT)]      # Using binary id, this column will help for selecting the correct protein marker for each host (subCT)
    df_bin['host_name'] = [
        "".join(["Host_",str(sum([int(s) for s in pos])),"_"]+[
           initials_sCT[n-1] for n in [int(pos[numb_pos])*(numb_pos+1) for numb_pos in range(len(subCT_markers))] if n!=0
        ]+[f"_sub{short_ct_name}"]) for pos in df_bin['list_pos']
    ]
    df_bin['Markers name'] = [
        ",".join([
           subCT_markers[n-1].replace(')','').replace('* (MV - ','_') for n in [int(pos[numb_pos])*(numb_pos+1) for numb_pos in range(len(subCT_markers))] if n!=0
        ]) for pos in df_bin['list_pos']
    ]
    return df_bin

This function is for calculating the proportion of subCTs in each sample.

In [18]:
def calculate_subCT_cell_proportions(adata_dicts,df_bin,subCT_markers,short_ct_name,ct_name="cells"):
    subCT_dict = {}
    for k, adata in adata_dicts.items():
        title_name = f"Sample {k} ({adata.n_obs} cells)"
        save_namefile_subs = f"{results_dir}sub{short_ct_name} cell proportions - {title_name}.csv"
        if os.path.exists(save_namefile_subs) and not override_existing_plots: 
            subCT_dict[k] = pd.read_csv(save_namefile_subs)
            continue
        length_CT = (adata.obs['clusters'] == ct_name).sum()
        adata_CT = adata[adata.obs['clusters'] == ct_name].copy()
        # df_CT = adata_CT.to_df()
        df_CT = pd.DataFrame(adata_CT.obsm["Positivity"], columns=pd.read_csv(f"{results_dir}Positivity column names.csv")["Positivity column names"].tolist())
        
        if length_CT!=0:
            dic_lenght_subCT = {}
            for name_host,pos in zip(df_bin['host_name'],df_bin['list_pos']):
                mask = pd.Series(True, index=df_CT.index)
                # print(mask)
                # print(df_CT.columns)
                for m, p in zip(subCT_markers, pos):
                    colname = df_CT.columns[df_CT.columns.str.contains(f"Positivity - {m}", regex=False)][0]
                    mask &= df_CT[colname] == int(p)
                df_filtered = df_CT[mask]
                dic_lenght_subCT[name_host] = len(df_filtered)
        else:
            dic_lenght_subCT = {name_host:0 for name_host in df_bin['host_name']}
        
        df_subCT = pd.DataFrame(data={
            "Host name":dic_lenght_subCT.keys(),
            "Total cells":dic_lenght_subCT.values()
        })
        if length_CT!=0:
            df_subCT["Proportion"] = [f"{(n/length_CT):.2%}" for n in df_subCT["Total cells"]]
        else:
            df_subCT["Proportion"] = [f"{(n):.2%}" for n in df_subCT["Total cells"]]

        df_temp = pd.concat([df_subCT, df_bin['Markers name']], axis=1)
        df_temp[df_temp["Total cells"]!=0].to_csv(save_namefile_subs)
        subCT_dict[k] = df_temp[df_temp["Total cells"]!=0]

        save_namefile_info = f"{results_dir}sub{short_ct_name} information - {title_name}.csv"
        if os.path.exists(save_namefile_info) and not override_existing_plots: continue    
        df_info = pd.DataFrame(data={"Information":[(", ").join(subCT_markers), length_CT]}, index=["Used markers", f"Total {ct_name} in sample"])
        df_info.to_csv(save_namefile_info)
    return subCT_dict
        

This code is for applying the previous functions.

In [19]:
df_bin = naming_subCTs(subCT_markers,initials_sCT,short_ct_name)

In [20]:
subCT_dict = calculate_subCT_cell_proportions(adata_dicts=adata_dicts, df_bin=df_bin, subCT_markers=subCT_markers, short_ct_name = short_ct_name, ct_name=ct_name)

In [21]:
adata_dicts["B2"]

AnnData object with n_obs × n_vars = 3478 × 32
    obs: 'clusters', 'only CAF cells', 'only Immune cells', 'only Tumor cells 1', 'only Tumor cells 2', 'leiden'
    uns: 'leiden', 'neighbors', 'pca', 'umap'
    obsm: 'ID_cell', 'Positivity', 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

## Common and unique subCT when considering treatment type

This function is for filtering phenotypes (subCT defined by specific protein markers) by average proportion greater or equal than 1% 

In [22]:
def filtering_phenotypes(df_tt,subCT_dict, dictionary):
    df_tt['Proportion Average'] = 0.0
    df_tt['Standard deviation'] = 0.0
    df_tt['Phenotype'] = "no_name"
    mean_list = []
    std_list = []
    values_list = []
    values_sample_name_list = []
    for host in df_tt.index.tolist():
        list_temp = []
        list_sample_name_temp = []
        for k, df_phenotype in subCT_dict.items():
            if k not in dictionary.keys(): continue
            if host in df_phenotype['Host name'].tolist():
                list_temp.append(float(df_phenotype['Proportion'][df_phenotype['Host name']==host].tolist()[0][:-1])/100)
                list_sample_name_temp.append(k)
                corresponding_pheno_list = df_phenotype['Markers name'][df_phenotype['Host name']==host].tolist()
                df_tt.loc[host, 'Phenotype'] = corresponding_pheno_list[0]
        mean_list.append(np.mean(list_temp))
        std_list.append(np.std(list_temp))
        values_list.append([val*100 for val in list_temp])
        values_sample_name_list.append([val for val in list_sample_name_temp])
    df_tt['Proportion Average'] = mean_list
    df_tt['Standard deviation'] = std_list
    df_tt['Proportions'] = values_list
    df_tt['Sample name where proportion comes from'] = values_sample_name_list
    df_tt = df_tt[df_tt['Proportion Average']>=0.01]
    df_tt = df_tt.sort_values(by="Proportion Average", ascending=False)
    return df_tt

This function is for counting phenotypes

In [23]:
def count_phenotypes(subCT_dict, dictionary):
    all_phenotypes = []
    for k, df_phenotype in subCT_dict.items():
        if k not in dictionary.keys(): continue
        for host in df_phenotype['Host name']:
            all_phenotypes.append(host)
    count_dict = {item:all_phenotypes.count(item) for item in set(all_phenotypes)}
    df_tt = pd.DataFrame(count_dict.values(),index=count_dict.keys(),columns=['Counts'])
    df_tt['Treatment'] = list(dictionary.values())[0]
    df_tt = filtering_phenotypes(df_tt, subCT_dict, dictionary)
    return df_tt

This function is for obtaining common and unique pehontypes per treatment type

In [24]:
def common_n_unique_phenotypes(df_phenotypes_ctrl, name_ctrl, df_phenotypes_trtmnt, name_trtmnt, short_ct_name):
    save_namefile_all = f"{results_dir}Common sub{short_ct_name}.csv"
    save_namefile_ctrl = f"{results_dir}Unique sub{short_ct_name} - {name_ctrl}.csv"
    save_namefile_trtmnt = f"{results_dir}Unique sub{short_ct_name} - {name_trtmnt}.csv"
    
    together_list = [host for host in df_phenotypes_ctrl.index.tolist() if host in df_phenotypes_trtmnt.index.tolist()]
    list_unique_ctrl = [host for host in df_phenotypes_ctrl.index.tolist() if host not in together_list]
    list_unique_trtmnt = [host for host in df_phenotypes_trtmnt.index.tolist() if host not in together_list]
    
    df_together = df_phenotypes_ctrl.loc[together_list].sort_values(by="Proportion Average", ascending=False)
    df_together = df_together[["Phenotype","Proportion Average","Standard deviation","Proportions"]].copy()
    df_together = df_together.rename(columns={"Proportion Average":f"{name_ctrl} %","Proportions":f"{name_ctrl} Prop Values"})
    df_together[f"{name_trtmnt} %"] = (df_phenotypes_trtmnt.loc[together_list]["Proportion Average"]*100).round(2)
    df_together[f"{name_trtmnt} Prop Values"] = df_phenotypes_trtmnt.loc[together_list]["Proportions"]
    df_together[f"{name_ctrl} %"] = (df_together[f"{name_ctrl} %"]*100).round(2)
    df_together = df_together.drop(columns=["Standard deviation"])
    df_together[f"Difference {name_trtmnt}-{name_ctrl}"] =  df_together[f"{name_trtmnt} %"] - df_together[f"{name_ctrl} %"]
    df_together['Interpretation'] = [f"Increase after {name_trtmnt}" if value>0 else f"Decrease after {name_trtmnt}" for value in df_together[f"Difference {name_trtmnt}-{name_ctrl}"]]
    df_together.to_csv(save_namefile_all)
    
    df_unique_ctrl = df_phenotypes_ctrl.loc[list_unique_ctrl].sort_values(by="Proportion Average", ascending=False)
    df_unique_ctrl = df_unique_ctrl[["Phenotype","Proportion Average","Treatment"]].copy()
    df_unique_ctrl = df_unique_ctrl.rename(columns={"Proportion Average":"%","Treatment":"Condition"})
    df_unique_ctrl["%"] = (df_unique_ctrl["%"]*100).round(2)
    df_unique_ctrl.to_csv(save_namefile_ctrl)

    df_unique_trtmnt = df_phenotypes_trtmnt.loc[list_unique_trtmnt].sort_values(by="Proportion Average", ascending=False)
    df_unique_trtmnt = df_unique_trtmnt[["Phenotype","Proportion Average","Treatment"]].copy()
    df_unique_trtmnt = df_unique_trtmnt.rename(columns={"Proportion Average":"%","Treatment":"Condition"})
    df_unique_trtmnt["%"] = (df_unique_trtmnt["%"]*100).round(2)
    df_unique_trtmnt.to_csv(save_namefile_trtmnt)
    
    return df_together, df_unique_ctrl, df_unique_trtmnt

This code is for applying the previous functions

In [25]:
df_phenotypes_TN = count_phenotypes(subCT_dict, dict_ctrl)
df_phenotypes_TNT = count_phenotypes(subCT_dict, dict_trtmnt)

In [26]:
df_together, df_unique_TN, df_unique_TNT = common_n_unique_phenotypes(df_phenotypes_ctrl = df_phenotypes_TN, name_ctrl = list(dict_ctrl.values())[0], df_phenotypes_trtmnt = df_phenotypes_TNT, name_trtmnt = list(dict_trtmnt.values())[0], short_ct_name=short_ct_name)

## Differences analysis (p-value)

This function is for calculating median in IQR

In [27]:
def median_iqr(values):
    q1, q3 = np.percentile(values, [25, 75])
    return np.median(values), q1, q3

This function is for putting stars for pvals in boxplots and captioning

In [28]:
def pval_to_stars(p):
    if p is None:
        return "ns"
    if p <= 1e-4:
        return "****"
    if p <= 1e-3:
        return "***"
    if p <= 1e-2:
        return "**"
    if p <= 5e-2:
        return "*"
    return "ns"

def annotate_pvals(ax, df, pairs, pvals, x="Phenotype", y="Proportion", hue="group",
                   hue_order=None, box_width=0.8, line_height=0.02, line_spacing=0.05,
                   text_offset=0.01, fontsize=12, color="k"):
    """
    ax: Matplotlib Axes (after seaborn boxplot was drawn)
    df: dataframe used to plot
    pairs: list of ((x_cat, hue1), (x_cat, hue2)) or ((x_cat1, hue1), (x_cat2, hue2))
    pvals: list of p-values aligned with pairs
    hue_order: optional list specifying hue order used by the plot (important to match offsets)
    """

    # determine unique x categories (in plot order)
    if pd.api.types.is_categorical_dtype(df[x]):
        x_categories = list(df[x].cat.categories)
    else:
        # keep the order seen in the plotting dataframe
        x_categories = list(dict.fromkeys(df[x].tolist()))

    # determine hue order
    if hue_order is None:
        if pd.api.types.is_categorical_dtype(df[hue]):
            hue_order = list(df[hue].cat.categories)
        else:
            hue_order = list(dict.fromkeys(df[hue].tolist()))

    n_hues = len(hue_order)

    # horizontal offsets for each hue around integer x positions
    # smaller step -> boxes closer; tune box_width to match seaborn/style
    max_offset = 0.2 * box_width
    if n_hues > 1:
        offsets = np.linspace(-max_offset*(n_hues-1), max_offset*(n_hues-1), n_hues)
    else:
        offsets = np.array([0.0])

    # track top y for each (x_category index) to stack multiple annotations
    used_top = {}  # key: (x_index_midpoint) -> current top y used

    # axis vertical span (for relative offsets)
    y_min, y_max = ax.get_ylim()
    y_span = y_max - y_min

    for (pair, p) in zip(pairs, pvals):
        (xa, ha), (xb, hb) = pair
        # indexes
        try:
            ix_a = x_categories.index(xa)
            ix_b = x_categories.index(xb)
        except ValueError:
            raise ValueError(f"Category {xa} or {xb} not found in x categories: {x_categories}")

        try:
            ih_a = hue_order.index(ha)
            ih_b = hue_order.index(hb)
        except ValueError:
            raise ValueError(f"Hue {ha} or {hb} not found in hue_order: {hue_order}")

        x1 = ix_a + offsets[ih_a]
        x2 = ix_b + offsets[ih_b]
        xmid = (x1 + x2) / 2.0

        # get max y among the two groups / categories to place the bracket above actual data
        sel = df[( (df[x] == xa) & (df[hue] == ha) ) | ( (df[x] == xb) & (df[hue] == hb) )]
        if sel.empty:
            # fallback to using global max
            ymax_pair = df[y].max()
        else:
            ymax_pair = sel[y].max()

        # base starting y (a bit above the data)
        pad = line_height * y_span
        y_base = ymax_pair + pad

        # stack if necessary (if there's already an annotation near xmid)
        key = (round(xmid, 3),)  # allow stacking per midpoint
        existing = used_top.get(key, y_base)
        # if the base is below existing, push up; otherwise start new at base
        if existing > y_base:
            y_val = existing + line_spacing * y_span
        else:
            y_val = y_base

        used_top[key] = y_val

        # small vertical step for the bracket ends
        dh = line_spacing * y_span

        # draw bracket
        ax.plot([x1, x1, x2, x2], [y_val, y_val+dh, y_val+dh, y_val], lw=1.0, c=color)

        # stars text
        stars = pval_to_stars(p)
        if stars == "ns":
            label = "ns"
        else:
            label = stars

        ax.text(xmid, y_val+dh+text_offset*y_span, label, ha='center', va='bottom', color=color, fontsize=fontsize)

    # optionally expand y-limits to make sure labels visible
    new_ymax = max(used_top.values()) + 2*line_spacing*y_span
    if new_ymax > y_max:
        ax.set_ylim(y_min, new_ymax)

def add_caption_and_legend(ax, caption, legend_text,
                           caption_x='center', fontsize=10,
                           caption_pad=0.01, legend_pad=0.02):
    """
    Place a centered caption and a left-aligned legend text below the x tick labels.
    ax: the Axes that contains the boxplot (e.g., returned from sns.boxplot)
    caption: string for the centered caption
    legend_text: string for the left-aligned annotation legend (can contain newlines)
    caption_x: 'center' or float in figure coordinates (0..1) for caption x position
    caption_pad: extra vertical padding (figure fraction) below the tick labels
    legend_pad: horizontal offset (figure fraction) left of the axes for legend text
    """
    fig = ax.figure

    # Force a draw so we can measure text extents
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    # Get axis position in figure coords
    ax_pos = ax.get_position()  # Bbox: x0, y0, width, height

    # Collect bounding boxes of visible x tick labels
    tick_labels = [lbl for lbl in ax.get_xticklabels() if lbl.get_text() != "" and lbl.get_visible()]
    if not tick_labels:
        # fallback: place texts relative to axes bbox
        tick_bottom_fig = ax_pos.y0
        tick_top_fig = ax_pos.y0 + 0.01
        label_height_fig = 0.01
    else:
        bboxes = [lbl.get_window_extent(renderer) for lbl in tick_labels]
        # ymin and ymax in display coords (pixels)
        min_y_display = min(b.y0 for b in bboxes)
        max_y_display = max(b.y1 for b in bboxes)

        # Convert to figure coordinates
        _, min_y_fig = fig.transFigure.inverted().transform((0, min_y_display))
        _, max_y_fig = fig.transFigure.inverted().transform((0, max_y_display))
        tick_bottom_fig = min_y_fig
        tick_top_fig = max_y_fig
        label_height_fig = max_y_fig - min_y_fig

    # Decide y position for texts (figure coords)
    caption_y = tick_bottom_fig + caption_pad + 0.5 * label_height_fig
    legend_y = caption_y  # same vertical line for legend

    # Ensure there is enough bottom margin in the figure; expand if needed
    # subplotpars.bottom expects a fraction of the figure height
    current_bottom = fig.subplotpars.bottom
    # keep a small safety margin
    needed_bottom = max(0.0, caption_y - 0.02)
    if needed_bottom < current_bottom:
        # reduce bottom to needed_bottom to create more space (smaller value -> more space)
        fig.subplots_adjust(bottom=max(0.0, needed_bottom))

    # Recompute caption_y after adjusting subplots (renderer may change)
    fig.canvas.draw()
    # Safety: if caption_x is 'center' place at axis center; otherwise use provided float
    if caption_x == 'center':
        caption_x_fig = ax_pos.x0 + ax_pos.width / 2.0
        caption_ha = 'center'
    else:
        caption_x_fig = float(caption_x)
        caption_ha = 'center' if 0.0 <= caption_x_fig <= 1.0 else 'left'

    # X for left-aligned legend: slightly left of the axis left edge
    legend_x_fig = max(0.0, ax_pos.x0 - legend_pad)

    # Finally draw texts in figure coordinates
    fig.text(caption_x_fig, caption_y, caption, ha=caption_ha, va='top', fontsize=fontsize)
    fig.text(legend_x_fig, legend_y, legend_text, ha='left', va='top', fontsize=fontsize)

This function is for plotting the table of common phenotypes

In [29]:
def plotting_table(df_together, name_ctrl, name_trtmnt, pvals, caption_text, short_ct_name, test_name = "test", dpi=300, bbox_inches='tight'):
    save_namefile = f"{plots_dir}Common sub{short_ct_name} proportion differences analysis - Bar plot - {test_name}.png"
    df_control = pd.DataFrame(df_together.iloc[0,2],columns=[df_together.index[0]])
    df_treat = pd.DataFrame(df_together.iloc[0,4],columns=[df_together.index[0]])
    for n in range(len(df_together.index)):
        df_control = pd.concat([df_control,pd.DataFrame(df_together.iloc[n,2],columns=[df_together.index[n]])],axis=1)
        df_treat = pd.concat([df_treat,pd.DataFrame(df_together.iloc[n,4],columns=[df_together.index[n]])],axis=1)
    df_control["group"] = name_ctrl
    df_treat["group"] = name_trtmnt
    
    df_long = pd.concat([df_control, df_treat], axis=0)
    df_long = df_long.melt(id_vars="group", var_name="Phenotype", value_name="Proportion")
    
    plt.figure(figsize=(8,7))
    ax = sns.boxplot(
        data=df_long,
        x="Phenotype",
        y="Proportion",
        hue="group",
        palette=["#d9d9d9","#a6c9ec"],
        showfliers=False
    )
    sns.stripplot(
        data=df_long,
        x="Phenotype",
        y="Proportion",
        hue="group",
        palette=["#d9d9d9","#a6c9ec"],
        dodge=True,
        alpha=0.5,
        linewidth=0.5
    )
    
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], labels[:2])

    pairs = [((valor,name_ctrl),(valor,name_trtmnt)) for valor in df_together.index]
    
    # annot = Annotator(ax, pairs, data=df_long, x="Phenotype", y="Proportion", hue="group")
    # annot.configure(test=None, text_format="star", loc="inside", line_offset=0.05)
    # annot.set_pvalues(pvals)  # p-values
    # annot.annotate()

    annotate_pvals(ax, df_long, pairs, pvals, x="Phenotype", y="Proportion", hue="group", hue_order=None, box_width=0.8)
    
    plt.title(f"Phenotype proportions: {name_ctrl} vs {name_trtmnt}")
    ax.tick_params(axis='x', rotation=90)
    caption = caption_text
    annotation_legend_text="" \
    "p-value annotation legend:\n\
    ns: 0.05 < p\n\
    *: 0.01 < p <= 0.05\n\
    **: 0.001 < p <= 0.01\n\
    ***: 0.0001 < p <= 0.001\n\
    ****: p <= 0.0001"
    # plt.text(7.5, -60.7, caption, ha='center', fontsize=10)
    # plt.text(-2, -65.8, annotation_legend_text, ha='left', fontsize=10)
    add_caption_and_legend(ax, caption, annotation_legend_text, fontsize=10, legend_pad=2/len(df_long["Phenotype"].unique()))
    plt.tight_layout()
    plt.savefig(save_namefile, dpi=dpi, bbox_inches=bbox_inches)
    plt.close()

This function is for obtaining table of common phenotypes proportions analysis

In [30]:
def common_phenotypes_differences_analysis(df_together, name_ctrl, name_trtmnt, short_ct_name):
    save_namefile = f"{results_dir}Common sub{short_ct_name} proportion differences analysis.csv"
    cell_types = df_together.index  # rows are cell types
    pvals = []
    pvals_m = []
    summary = []
    
    for cell in cell_types:
        ctrl_vals = df_together.loc[cell,f"{name_ctrl} Prop Values"]
        treat_vals = df_together.loc[cell,f"{name_trtmnt} Prop Values"]
        
        # mean ± SD
        ctrl_mean, ctrl_sd = np.mean(ctrl_vals), np.std(ctrl_vals, ddof=1)
        treat_mean, treat_sd = np.mean(treat_vals), np.std(treat_vals, ddof=1)
        
        ctrl_med, ctrl_q1, ctrl_q3 = median_iqr(ctrl_vals)
        treat_med, treat_q1, treat_q3 = median_iqr(treat_vals)
    
        # Welch t-test
        sta_m, pval_m = ttest_ind(ctrl_vals, treat_vals, equal_var=False)
        stat, pval = mannwhitneyu(ctrl_vals, treat_vals, alternative="two-sided")
        pvals.append(pval)
        pvals_m.append(pval_m)
        
        summary.append({
            "Cell type": cell,
            "Control (mean ± SD)": f"{ctrl_mean:.2f} ± {ctrl_sd:.2f}",
            "Treatment (mean ± SD)": f"{treat_mean:.2f} ± {treat_sd:.2f}",
            "Control (median [IQR])": f"{ctrl_med:.2f} [{ctrl_q1:.2f}–{ctrl_q3:.2f}]",
            "Treatment (median [IQR])": f"{treat_med:.2f} [{treat_q1:.2f}–{treat_q3:.2f}]",
            "Mann–Whitney p-value": pval,
            "Raw p-value": pval_m
        })
    
    # FDR correction
    _, fdr_pvals, _, _ = multipletests(pvals, method="fdr_bh")
    _, fdr_pvals_m, _, _ = multipletests(pvals_m, method="fdr_bh")
    
    # Add corrected p-values
    for i, row in enumerate(summary):
        row["FDR-corrected M-W p-value"] = fdr_pvals[i]
        row["FDR-corrected RAW p-value"] = fdr_pvals_m[i]
    
    results_table = pd.DataFrame(summary)
    results_table.to_csv(save_namefile)

    plotting_table(df_together, name_ctrl, name_trtmnt, pvals,   caption_text = "p-values from Mann–Whitney U test. Significance was assessed\nusing FDR correction across cell types;\nonly results with FDR < 0.05 were considered significant.", short_ct_name=short_ct_name, test_name = "Mann–Whitney") # t test
    plotting_table(df_together, name_ctrl, name_trtmnt, pvals_m, caption_text = "p-values from Welch t-test. Significance was assessed\nusing FDR correction across cell types;\nonly results with FDR < 0.05 were considered significant.",        short_ct_name=short_ct_name, test_name = "t-test") # Mann–Whitney

This code is for applying the previous function

In [31]:
common_phenotypes_differences_analysis(df_together = df_together, name_ctrl = list(dict_ctrl.values())[0], name_trtmnt = list(dict_trtmnt.values())[0], short_ct_name=short_ct_name)

/opt/modules/pycirclize/pycirclize_env/lib/python3.10/site-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/modules/pycirclize/pycirclize_env/lib/python3.10/site-packages/numpy/_core/_methods.py:210: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/modules/sklearn/sklearn-env/lib/python3.10/site-packages/scipy/stats/_stats_py.py:1113: RuntimeWarning: divide by zero encountered in divide
  var *= np.divide(n, n-ddof)  # to avoid error on division by zero
/opt/modules/sklearn/sklearn-env/lib/python3.10/site-packages/scipy/stats/_stats_py.py:1113: RuntimeWarning: invalid value encountered in scalar multiply
  var *= np.divide(n, n-ddof)  # to avoid error on division by zero
/tmp/ipykernel_3427749/4198858791.py:26: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.Ca